In [20]:
import pandas as pd
import numpy as np 
import seaborn as sns
from typing import Literal
from dataclasses import dataclass

In [21]:
BB_PERIOD = 20
SMA_PERIOD = 20
ATR_PERIOD = 9
STD_MULTIPLES = (2.0, 3.0)
StopMode = Literal['candle','atr']
PATH_CSV = 'win_5m.csv'

In [22]:
#Classes
@dataclass
class Trade:
    direction: Literal['long','short']
    entry_idx: int
    entry_time: pd.Timestamp
    entry_price: float
    stop_price: float
    target_sma: float
    exit_idx: int
    exit_time: pd.Timestamp
    exit_price: float
    exit_reason: Literal['target','stop','end']
    pnl: float
    bars_held: int

@dataclass
class BacktestStats:
    label: str
    n_trades: int
    winrate: float
    payoff: float
    avg_win: float
    avg_loss: float
    expected_value: float
    frequency_per_1000_bars: float
    max_drawdown_pct: float
    max_drawdown_nominal: float
    max_win_streak: int
    max_loss_streak: int
    equity: np.ndarray
    drawdown_pct: np.ndarray
    trades: list[Trade]


In [23]:
def load_data(path) -> pd.DataFrame:
    data = pd.read_csv(path, sep=';', index_col='Data')
    data.index = pd.to_datetime(data.index)
    return data

df = load_data(PATH_CSV)
df.head()

C:\Users\Bruno\AppData\Local\Temp\ipykernel_3180\1233226254.py:3: UserWarning: Parsing dates in %d/%m/%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  data.index = pd.to_datetime(data.index)


,Abertura,Maxima,Minima,Fechamento
Data,,,,
2026-05-21 18:20:00,178830,179055,178700,178700
2026-05-21 18:15:00,178850,178930,178730,178825
2026-05-21 18:10:00,178865,178915,178745,178850
2026-05-21 18:05:00,178935,178980,178755,178865
2026-05-21 18:00:00,179110,179140,178930,178945


In [24]:
def add_indicators(df):
    out = df.copy()
    c = out['Fechamento']

    mid = c.rolling(BB_PERIOD).mean()
    std = c.rolling(BB_PERIOD).std(ddof=1) #assuming this is just a sample, obviously
    out['bb_mid'] = mid
    for m in STD_MULTIPLES:
        out[f'bb_upper_{m}'] = mid + m * std
        out[f'bb_lower_{m}'] = mid - m * std

    out['sma20'] = c.rolling(SMA_PERIOD).mean()

    prev_close = c.shift(1)
    tr = pd.concat(
        [
            out['Maxima'] - out['Minima'],
            (out['Maxima'] - prev_close).abs(),
            (out['Minima'] - prev_close).abs(),
        ],
        axis=1,
    ).max(axis=1)
    out['atr'] = tr.rolling(ATR_PERIOD).mean()
    return out

add_indicators(df=load_data(PATH_CSV)).tail()

C:\Users\Bruno\AppData\Local\Temp\ipykernel_3180\1233226254.py:3: UserWarning: Parsing dates in %d/%m/%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  data.index = pd.to_datetime(data.index)


,Abertura,Maxima,Minima,Fechamento,bb_mid,bb_upper_2.0,bb_lower_2.0,bb_upper_3.0,bb_lower_3.0,sma20,atr
Data,,,,,,,,,,,
2017-01-27 09:20:00,66425,66470,66390,66415,66476.75,66572.076309,66381.423691,66619.739464,66333.760536,66476.75,77.777778
2017-01-27 09:15:00,66445,66450,66395,66425,66477.50,66570.830200,66384.169800,66617.495300,66337.504700,66477.50,72.777778
2017-01-27 09:10:00,66460,66480,66420,66445,66481.25,66561.504529,66400.995471,66601.631794,66360.868206,66481.25,69.444444
2017-01-27 09:05:00,66410,66475,66390,66455,66482.75,66559.628236,66405.871764,66598.067354,66367.432646,66482.75,70.555556
2017-01-27 09:00:00,66600,66660,66380,66415,66481.25,66562.287678,66400.212322,66602.806517,66359.693483,66481.25,96.111111


In [25]:
def simulate_trade(df, entry_i, direction, entry_price, stop_mode):
    row = df.iloc[entry_i]
    atr = row['atr']
    if pd.isna(atr):
        return None
    
    if stop_mode == 'candle':
        stop = float(row['low']) if direction == 'long' else float(row['high'])
    else:
        stop = entry_price - atr if direction == 'long' else entry_price + atr

    if direction == 'long' and stop >= entry_price:
        return None
    if direction == 'short' and stop <= entry_price:
        return None

    n = len(df)
    for j in range(entry_i + 1, n):
        bar = df.iloc[j]
        sma = bar['sma20']
        if pd.isna(sma):
            continue

        h, l = float(bar['high']), float(bar['low'])

        if direction == 'long':
            stop_hit = l <= stop
            target_hit = h >= sma
            if stop_hit:
                return _close_trade(df, entry_i, j, direction, entry_price, stop, sma, 'stop', stop)
            if target_hit:
                return _close_trade(df, entry_i, j, direction, entry_price, stop, sma, 'target', sma)
        else:
            stop_hit = h >= stop
            target_hit = l <= sma
            if stop_hit:
                return _close_trade(df, entry_i, j, direction, entry_price, stop, sma, 'stop',stop)
            if target_hit:
                return _close_trade(df, entry_i, j, direction, entry_price, stop, sma, 'target',)
    
    last = df.iloc[-1]
    last_price = float(last['close'])
    return _close_trade(df, entry_i, n-1, direction, entry_price, stop, float(last['sma20']), 'end',last_price)

def _close_trade(df, entry_i, exit_i, direction, entry_price, stop, target_sma, reason, exit_price):
    pnl = (exit_price - entry_price) if direction == 'long' else (entry_price - exit_price)
    return Trade(
        direction= direction,
        entry_idx= entry_i,
        entry_time= df.index[entry_i],
        entry_price= entry_price,
        stop_price= stop,
        target_sma = target_sma,
        exit_idx= exit_i,
        exit_time= df.index[exit_i],
        exit_price= exit_price,
        exit_reason= reason,
        pnl= pnl,
        bars_held= exit_i - entry_i
    )

In [28]:
def backtest(df, std_mult, stop_mode, initial_equity) -> BacktestStats:
    upper = f'bb_upper_{std_mult}'
    lower = f'bb_lower_{std_mult}'
    label = f'BB {std_mult:.0f}std | Sl ={"candle" if stop_mode == "candle" else "1ATR"}'

    trades: list[Trade] = []
    i = max(BB_PERIOD, ATR_PERIOD, SMA_PERIOD) # skipping lines with NaN for while start at 20
    n = len(df)

    while i < n - 1:
        row = df.iloc[i]
        if pd.isna(row[upper]) or pd.isna(row[lower]) or pd.isna(row['sma20']):
            i += 1
            continue
        
        close = float(row['close'])
        direction = None
        if close > float(row[upper]):
            direction = 'short'
        elif close < float(row[lower]):
            direction = 'long'
        
        if direction is None:
            i += 1
            continue

        entry_price = close
        trade = simulate_trade(df, i, direction, entry_price, stop_mode)
        if trade is None:
            i += 1
            continue

        trades.append(trade)
        i = trade.exit_idx + 1
    
    return _compute_stats(trades, label, n, initial_equity)

def _compute_stats(trades, label, n_bars, initial_equity) -> BacktestStats:
    if not trades:
        return BacktestStats(
            label=label, n_trades=0, winrate=0.0, payoff=0.0, avg_win=0.0, avg_loss=0.0,
            expected_value=0.0, frequency_per_1000_bars=0.0, max_drawdown_pct=0.0,
            max_drawdown_nominal=0.0, max_win_streak=0, max_loss_streak=0,
            equity=np.array([initial_equity]), drawdown_pct=np.array([0.0]), trades=[],
        )
    
    pnls = np.array([t.pnl for t in trades])
    wins = pnls[pnls > 0]
    losses = pnls[pnls < 0]

    winrate = len(wins) / len(pnls)
    avg_win = float(wins.mean()) if len(wins) else 0.0
    avg_loss = float(np.abs(losses.mean())) if len(losses) else 0.0
    payoff = avg_win / avg_loss if avg_loss > 0 else np.inf if avg_win > 0 else 0.0
    print('avgerage loss',avg_loss)

    equity = initial_equity + np.cumsum(pnls)
    equity_curve = np.concatenate([[initial_equity], equity])
    peak = np.maximum.accumulate(equity_curve)
    dd_nom = equity_curve - peak
    with np.errstate(divide='ignore', invalid='ignore'):
        dd_pct = np.where(peak > 0, dd_nom/peak * 100.0, 0.0)
    
    max_wins, max_losses, current_wins, current_losses - 0, 0, 0, 0
    for p in pnls:
        if p > 0:
            current_wins += 1
            current_losses = 0
            max_wins = max(max_wins, current_wins)
        elif p < 0:
            current_losses += 1
            current_wins = 0
            max_losses = max(max_losses, current_losses)
        else:
            current_wins = current_losses = 0
    
    return BacktestStats(
        label=label,
        n_trades=len(trades),
        winrate=winrate,
        payoff=float(payoff) if np.isfinite(payoff) else 999.0,
        avg_win=avg_win,
        avg_loss=avg_loss,
        expected_value=float(pnls.mean()),
        frequency_per_1000_bars=len(trades) / n_bars * 1000,
        max_drawdown_pct=float(dd_pct.min()),
        max_drawdown_nominal=float(dd_nom.min()),
        max_win_streak=max_wins,
        max_loss_streak=max_losses,
        equity=equity_curve,
        drawdown_pct=dd_pct,
        trades=trades,
    )
